# Segmentación de tumores cerebrales en MRI con TDA
## Descripción de los datos

## Descripción del Conjunto de Datos

El conjunto de datos utilizado consiste en imágenes de resonancia magnética (MRI) cerebral, preprocesadas y almacenadas en formato HDF5 (.h5). Cada muestra corresponde a un volumen 3D cerebral con **4 canales** de información, representando distintas modalidades de imagen médica (T1, T1ce, T2 y FLAIR), utilizadas comúnmente en tareas de segmentación de tumores cerebrales. El conjunto completo contiene **369 resonancias cerebrales**, cada una de las cuales ha sido seccionada en cortes axiales individuales. Estos cortes han sido organizados en un total de **155 archivos HDF5 por resonancia**.

Cada canal de imagen corresponde a una modalidad diferente de resonancia magnética:
1. **T1**: Imagen ponderada en T1 (usada para observar la anatomía general del cerebro).
2. **T1Gd**: Imagen ponderada en T1 con contraste (utilizada para observar mejor los tumores y la lesión cerebral).
3. **T2**: Imagen ponderada en T2 (usada para identificar zonas de edema o inflamación).
4. **T2-FLAIR**: Imagen ponderada en FLAIR (utilizada para resaltar lesiones cerebrales y tumores, especialmente en el área de la sustancia blanca).

Este dataset se tomó de [Kaggle](https://www.kaggle.com/datasets/awsaf49/brats2020-training-data?select=BraTS20+Training+Metadata.csv) y proviene del desafío **BraTS2020 (Brain Tumor Segmentation año 2020)**, ampliamente utilizado para la evaluación de modelos de aprendizaje profundo en tareas médicas de segmentación 3D.

In [1]:
import h5py
import numpy as np
import os

# Ruta a los archivos .h5 (suponiendo que los cortes están en una carpeta)
folder_path = 'data/BraTS2020_training_data/content/data'  # Ruta a la carpeta donde están los archivos .h5
files = os.listdir(folder_path) # Obtener la lista de archivos

# Verifica que todos los archivos sean .h5 y tomamos solo los primeros 158 archivos9
h5_files = [f for f in files[0:157] if f.endswith('.h5')]

# Ordenar los archivos primero por volumen y luego por slice
sorted_h5_files = sorted(h5_files, key=lambda f: (int(f.split('_')[1]), int(f.split('_')[3].split('.')[0])))

# Listado de cortes 2D
volume_data_brain = []
volume_data_tumor = []

# Cargar cada archivo .h5 y extraer el corte 2D
for f in sorted_h5_files:
    with h5py.File(os.path.join(folder_path, f), 'r') as file:
        # La imagen 2D está almacenada en la clave 'image', el tumor en la clave 'mask'.
        brain_slice = file['image'][:]
        tumor_slice = file['mask'][:]

        volume_data_brain.append(brain_slice)
        volume_data_tumor.append(tumor_slice)

# Apilar los cortes 2D a lo largo del eje Z (creando el volumen 3D)
volume_3d_brain = np.stack(volume_data_brain, axis=-1)
volume_3d_tumor = np.stack(volume_data_tumor, axis=-1)

print("Forma del volumen 3D del cerebro:", volume_3d_brain.shape)
print("Forma del volumen 3D del tumor:", volume_3d_tumor.shape)

Forma del volumen 3D del cerebro: (240, 240, 4, 154)
Forma del volumen 3D del tumor: (240, 240, 3, 154)


La media y desviación estándar de los datos que conforman la imagen sugieren que las señales del MRI paraon por un proceso de estandarización, por lo que aplicaremos un proceso de escalado para poder representar las señales como imágenes en escala de grises.

In [54]:
import pandas as pd

def volume_to_dataframe(volume:np.array, column_names:list[str]) -> pd.DataFrame:
    """
    Convierte un volumen multidimensional de imagen en un DataFrame de Pandas,
    aplanando cada canal como una columna.

    :param volume: np.array
        Arreglo NumPy de 4 dimensiones con forma (alto, ancho, canales, profundidad),
        donde cada canal representa una modalidad o tipo de imagen distinta.
    :param column_names: list of str
        Lista con los nombres de las columnas (uno por canal). Debe coincidir en longitud
        con el número de canales en el volumen (eje 2).
    :return: pd.DataFrame
        DataFrame con una columna por canal, en el cual cada fila representa un píxel
        a lo largo de todas las imágenes (slices).
    """
    flattened_channels = [np.ravel(volume[:,:,column,:]) for column in range(len(column_names))]
    return pd.DataFrame(np.asarray(flattened_channels).T, columns=column_names)

brain_channels = ['T1', 'T1Gd', 'T2', 'T2-FLAIR']

volume_3d_brain_df = volume_to_dataframe(volume_3d_brain, column_names=brain_channels)
volume_3d_brain_df.describe()

,T1,T1Gd,T2,T2-FLAIR
count,8.870400e+06,8.870400e+06,8.870400e+06,8.870400e+06
mean,1.879401e-16,6.110873e-17,-1.169371e-16,7.238718e-17
std,9.466277e-01,9.466277e-01,9.466277e-01,9.466277e-01
min,-5.152460e-01,-5.818055e-01,-5.751521e-01,-5.332813e-01
25%,-4.782978e-01,-5.134370e-01,-5.004875e-01,-4.801020e-01
50%,-2.991492e-01,-3.078841e-01,-2.949400e-01,-2.887100e-01
75%,-8.316670e-03,-8.332043e-03,0.000000e+00,-8.319255e-03
max,1.327499e+02,1.310595e+02,1.467040e+02,1.273901e+02


In [55]:
def scale_images_by_channel(image: np.array):
    """
    Escala los valores de cada canal de una imagen 3D a un rango de [0, 255].

    Este método normaliza los valores de cada canal independientemente usando min-max scaling,
    y convierte los resultados a tipo `uint8`, adecuado para visualización de imágenes.

    :param image: np.ndarray
        Arreglo NumPy de forma (H, W, C, N), donde:
        - H: altura de la imagen,
        - W: ancho de la imagen,
        - C: número de canales,
        - N: número de imágenes (o slices).
    :return: np.ndarray
        Arreglo escalado con forma (H, W, C, N) y tipo `uint8`.
    """
    scaled_images = []
    for channel in range(image.shape[2]):
        scaled_image = (image[:,:,channel,:] - np.min(image[:,:,channel,:])) * 255 / (np.max(image[:,:,channel,:]) -np.min(image[:,:,channel,:]))
        scaled_image = scaled_image.astype(np.uint8)
        scaled_images.append(scaled_image)
    return np.moveaxis(np.array(scaled_images), 0, 2)

volume_3d_brain_scaled = scale_images_by_channel(volume_3d_brain)
volume_3d_brain_df = volume_to_dataframe(volume_3d_brain_scaled, column_names=brain_channels)
volume_3d_brain_df.describe()

,T1,T1Gd,T2,T2-FLAIR
count,8.870400e+06,8.870400e+06,8.870400e+06,8.870400e+06
mean,6.005646e-01,8.043034e-01,5.714500e-01,7.575637e-01
std,1.747251e+00,1.772337e+00,1.586588e+00,1.822290e+00
min,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
25%,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
50%,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
75%,0.000000e+00,1.000000e+00,0.000000e+00,1.000000e+00
max,2.540000e+02,2.550000e+02,2.550000e+02,2.550000e+02


In [56]:
tumor_channels = ['ET', 'ED', 'NCR']

volume_3d_tumor_df = volume_to_dataframe(volume_3d_tumor, column_names=tumor_channels)
volume_3d_tumor_df.describe()

,ET,ED,NCR
count,8.870400e+06,8.870400e+06,8.870400e+06
mean,5.319264e-03,6.417185e-03,3.383726e-03
std,7.273905e-02,7.984989e-02,5.807130e-02
min,0.000000e+00,0.000000e+00,0.000000e+00
25%,0.000000e+00,0.000000e+00,0.000000e+00
50%,0.000000e+00,0.000000e+00,0.000000e+00
75%,0.000000e+00,0.000000e+00,0.000000e+00
max,1.000000e+00,1.000000e+00,1.000000e+00


In [57]:
volume_3d_tumor_scaled = scale_images_by_channel(volume_3d_tumor)
volume_3d_tumor_df = volume_to_dataframe(volume_3d_tumor_scaled, column_names=tumor_channels)
volume_3d_tumor_df.describe()

,ET,ED,NCR
count,8.870400e+06,8.870400e+06,8.870400e+06
mean,1.356412e+00,1.636382e+00,8.628500e-01
std,1.854846e+01,2.036172e+01,1.480818e+01
min,0.000000e+00,0.000000e+00,0.000000e+00
25%,0.000000e+00,0.000000e+00,0.000000e+00
50%,0.000000e+00,0.000000e+00,0.000000e+00
75%,0.000000e+00,0.000000e+00,0.000000e+00
max,2.550000e+02,2.550000e+02,2.550000e+02


In [64]:
import napari

# volumen_3d: (240, 240, 4, 154) del cerebro
# napari espera el volumen como (profundidad, altura, ancho)
volume_brain_for_napari = np.moveaxis(volume_3d_brain, [2, 3], [0, 1])  # (4, 154, 240, 240)
volume_tumor_for_napari = np.moveaxis(volume_3d_tumor, [2, 3], [0, 1])

# Crear el visor de Napari en 3D (ndisplay=3)
brain_3d_viewer = napari.Viewer(ndisplay=3)

# Añadir la imagen 3D con renderizado isovolumétrico para el modelo 3D
for i, vol in enumerate(volume_brain_for_napari):
    layer = brain_3d_viewer.add_image(
        data=vol,
        name=brain_channels[i],
        colormap='gray',
        rendering='iso',
        iso_threshold=1.5,
        contrast_limits=[1, 10],
        blending='opaque'
    )

tumor_3d_viewer = napari.Viewer(ndisplay=3)

for i, vol in enumerate(volume_tumor_for_napari):
    layer = tumor_3d_viewer.add_image(
        data=vol,
        name=tumor_channels[i],
        colormap='gray',
        rendering='iso',
        iso_threshold=1.5,
        contrast_limits=[1, 10],
        blending='opaque'
    )

napari.run()

In [66]:
# Crear el visor de Napari por slices
brain_2d_viewer = napari.Viewer(ndisplay=2)

# Añadir la imagen 2D para las slices (en cada canal) con renderizado normal
for i, vol in enumerate(volume_brain_for_napari):
    layer = brain_2d_viewer.add_image(
        data=vol,
        name=brain_channels[i],
        colormap='inferno',
        rendering='translucent',
    )
    layer.reset_contrast_limits()  # Activar autocontraste continuo
tumor_2d_viewer = napari.Viewer(ndisplay=2)

for i, vol in enumerate(volume_tumor_for_napari):
    layer = tumor_2d_viewer.add_image(
        data=vol,
        name=tumor_channels[i],
        colormap='inferno',
        rendering='translucent',
    )
    layer.reset_contrast_limits()  # Activar autocontraste continuo


napari.run()
